In [1]:
# Imports
import gc
import json
import time
from pathlib import Path

import cv2
import ffmpeg
import numpy as np
import torch
from faster_whisper import WhisperModel
from PIL import Image
from rapidocr_onnxruntime import RapidOCR
from transformers import AutoModelForCausalLM, AutoProcessor

In [2]:
# Paths and settings
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
VIDEO_PATH = RAW_DIR / "1183.mp4"
AUDIO_PATH = PROCESSED_DIR / f"{VIDEO_PATH.stem}_audio.wav"
OUTPUT_PATH = PROCESSED_DIR / f"{VIDEO_PATH.stem}_transcript.json"

SAMPLE_FPS = 1         # candidate frames checked per second of video
DIFF_THRESHOLD = 8.0   # mean pixel change (0-255) vs last kept frame to count as a new scene
OCR_MIN_SCORE = 0.5    # drop low-confidence OCR text

assert VIDEO_PATH.exists(), f"missing {VIDEO_PATH}"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Read video info (duration, fps) with ffprobe
probe = ffmpeg.probe(str(VIDEO_PATH))
video_stream = next(s for s in probe["streams"] if s["codec_type"] == "video")
num, den = video_stream["avg_frame_rate"].split("/")
VIDEO_FPS = int(num) / int(den)
VIDEO_DURATION = float(probe["format"]["duration"])
print(f"{VIDEO_PATH.name}: {VIDEO_DURATION:.1f}s @ {VIDEO_FPS:.2f} fps")

1183.mp4: 60.2s @ 23.98 fps


In [4]:
# AUDIO 1/3: extract the audio track with ffmpeg (mono, 16 kHz wav, what Whisper expects)
(
    ffmpeg
    .input(str(VIDEO_PATH))
    .output(str(AUDIO_PATH), vn=None, ac=1, ar=16000)
    .overwrite_output()
    .run(quiet=True)
)
print("saved", AUDIO_PATH)

saved data/processed/1183_audio.wav


In [5]:
# AUDIO 2/3: transcribe the audio with faster-whisper (same setup as audio/transcript)
whisper_model = WhisperModel(
    "small",
    device="cpu",
    compute_type="int8",
)

segments, info = whisper_model.transcribe(str(AUDIO_PATH))
segments = list(segments)
print(info)

audio_segments = [
    {
        "segment_id": segment.id,
        "start": round(segment.start, 2),
        "end": round(segment.end, 2),
        "text": segment.text.strip(),
        "avg_logprob": segment.avg_logprob,
        "no_speech_prob": segment.no_speech_prob,
    }
    for segment in segments
]

for s in audio_segments:
    print(f"[{s['start']:.2f}s -> {s['end']:.2f}s] {s['text']}")

TranscriptionInfo(language='en', language_probability=0.9990934133529663, duration=60.1165, duration_after_vad=60.1165, all_language_probs=[('en', 0.9990934133529663), ('pt', 0.0002138775889761746), ('cy', 0.00017042313993442804), ('la', 0.00012160953338025138), ('de', 5.18849010404665e-05), ('ja', 3.6576886486727744e-05), ('zh', 3.5570537875173613e-05), ('ko', 3.52578645106405e-05), ('ar', 3.517833101795986e-05), ('fr', 2.3618740669917315e-05), ('es', 1.7920614482136443e-05), ('pl', 1.7428792489226907e-05), ('nn', 1.4957819985284004e-05), ('ms', 1.4277045920607634e-05), ('id', 1.3869865142623894e-05), ('el', 1.0694848242565058e-05), ('vi', 7.585635103168897e-06), ('tr', 7.561654911114601e-06), ('hu', 7.372147592832334e-06), ('mi', 6.949189810256939e-06), ('nl', 6.587120878975838e-06), ('km', 6.438262516894611e-06), ('sv', 6.0639781622739974e-06), ('ru', 4.881761924480088e-06), ('haw', 4.377085133455694e-06), ('sw', 3.7461620649992255e-06), ('it', 3.6876258491247427e-06), ('fi', 2.6350

In [6]:
# AUDIO 3/3: free the Whisper model before loading the vision models (limited RAM)
del whisper_model
gc.collect()

0

In [7]:
# VISUAL 1/4: load Florence-2 (caption, object detection) and RapidOCR (text)
MODEL_ID = "microsoft/Florence-2-base"
DEVICE = "cpu"

florence = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float32,
    attn_implementation="eager",
).to(DEVICE).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

ocr_engine = RapidOCR()
print("models loaded")

models loaded


In [8]:
# VISUAL 2/4: per-frame processing (caption + objects + OCR)
def run_florence(task, image):
    """Run one Florence-2 task prompt and return the parsed result."""
    inputs = processor(text=task, images=image, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        ids = florence.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=128,
            num_beams=1,  # greedy: much faster on CPU than beam search
            do_sample=False,
            use_cache=False,  # avoids a cache bug with the pinned remote code
        )
    text = processor.batch_decode(ids, skip_special_tokens=False)[0]
    return processor.post_process_generation(
        text, task=task, image_size=(image.width, image.height)
    )


def run_ocr(frame_bgr):
    """Run RapidOCR on a frame and return text with box and confidence."""
    result, _ = ocr_engine(frame_bgr)
    items = []
    for polygon, text, score in result or []:
        if float(score) < OCR_MIN_SCORE:
            continue
        xs = [p[0] for p in polygon]
        ys = [p[1] for p in polygon]
        items.append({
            "text": text,
            "score": round(float(score), 3),
            "bbox": [round(min(xs)), round(min(ys)), round(max(xs)), round(max(ys))],
        })
    return items


def process_frame(frame_bgr):
    """Describe one frame: caption, detected objects and on-screen text."""
    image = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    caption = run_florence("<CAPTION>", image)["<CAPTION>"]
    od = run_florence("<OD>", image)["<OD>"]
    return {
        "caption": caption,
        "objects": [
            {"label": label, "bbox": [round(v) for v in box]}
            for label, box in zip(od["labels"], od["bboxes"])
        ],
        "ocr": run_ocr(frame_bgr),
    }

In [9]:
# VISUAL 3/4: pick keyframes (sample at SAMPLE_FPS, keep a frame only if the scene changed)
def iter_keyframes(video_path, sample_fps, diff_threshold):
    """Yield (timestamp_sec, frame_bgr) for frames that differ from the last kept one."""
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    step = max(1, round(fps / sample_fps))
    last_small = None
    idx = 0
    while cap.grab():  # grab() is cheap; only decode (retrieve) the sampled frames
        if idx % step == 0:
            ok, frame = cap.retrieve()
            if ok:
                small = cv2.cvtColor(cv2.resize(frame, (64, 36)), cv2.COLOR_BGR2GRAY)
                small = small.astype(np.float32)
                if last_small is None or np.abs(small - last_small).mean() > diff_threshold:
                    last_small = small
                    yield idx / fps, frame
        idx += 1
    cap.release()

In [10]:
# VISUAL 4/4: run the vision pipeline on every keyframe
raw_frames = []
t0 = time.time()
for t, frame in iter_keyframes(VIDEO_PATH, SAMPLE_FPS, DIFF_THRESHOLD):
    result = process_frame(frame)
    raw_frames.append((t, result))
    print(f"[{t:6.1f}s] frame {len(raw_frames)} | {time.time() - t0:.0f}s elapsed | {result['caption']}")

# each visual segment lasts until the next keyframe (the last one until the video ends)
starts = [round(t, 2) for t, _ in raw_frames]
ends = starts[1:] + [round(VIDEO_DURATION, 2)]
visual_segments = [
    {"segment_id": i, "start": start, "end": end, **result}
    for i, (start, end, (_, result)) in enumerate(zip(starts, ends, raw_frames), start=1)
]
print(f"{len(visual_segments)} visual segments")

/home/ubuntu/DEProj_Video_Analytics_Platform/local/.env/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:677: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


[   0.0s] frame 1 | 14s elapsed | the ted logo on a white background
[   1.0s] frame 2 | 28s elapsed | A red ted logo on a white background.
[   2.0s] frame 3 | 40s elapsed | a red and black background with a lot of red dots
[   3.0s] frame 4 | 52s elapsed | a red moon in the dark sky
[   4.0s] frame 5 | 67s elapsed | A man standing on a stage in front of a purple curtain.
[   5.0s] frame 6 | 81s elapsed | A man standing on a stage with his arms outstretched.
[   6.0s] frame 7 | 93s elapsed | A man standing on a stage in front of a purple curtain.
[   7.0s] frame 8 | 105s elapsed | A man standing on a stage holding a cell phone.
[   8.0s] frame 9 | 121s elapsed | A man standing on a stage in front of a purple curtain.
[   9.0s] frame 10 | 136s elapsed | A man standing on a stage giving a presentation.
[  10.0s] frame 11 | 150s elapsed | A man standing in front of a large vase.
[  11.0s] frame 12 | 165s elapsed | A man standing in front of a large vase on a stage.
[  12.0s] frame 13 | 1

In [11]:
# Merge audio and visual onto one timeline: for each visual segment, the speech that overlaps it
def overlaps(a_start, a_end, b_start, b_end):
    return a_start < b_end and b_start < a_end


timeline = [
    {
        "start": v["start"],
        "end": v["end"],
        "speech": " ".join(
            a["text"] for a in audio_segments
            if overlaps(a["start"], a["end"], v["start"], v["end"])
        ),
        "caption": v["caption"],
        "ocr_text": " ".join(o["text"] for o in v["ocr"]),
    }
    for v in visual_segments
]

In [12]:
# Save everything to data/processed as one JSON
output = {
    "video": VIDEO_PATH.name,
    "duration": round(VIDEO_DURATION, 2),
    "fps": round(VIDEO_FPS, 2),
    "language": info.language,
    "audio_segments": audio_segments,
    "visual_segments": visual_segments,
    "timeline": timeline,
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Saved {len(audio_segments)} audio / {len(visual_segments)} visual segments to {OUTPUT_PATH}")

Saved 12 audio / 33 visual segments to data/processed/1183_transcript.json
